# 08 - Text model: fine-tuning PhoBERT

Predicts the price from the listing text only, with a pretrained Vietnamese language model
([`vinai/phobert-base-v2`](https://huggingface.co/vinai/phobert-base-v2)).

**Question:** does a model that understands Vietnamese get more out of the listing text than TF-IDF word counts (notebook 06)?

PhoBERT is trained on the full training set and scored on the full test set, the same rows as notebooks 04-07.
The whole training loop is handled by the Hugging Face `Trainer`.

**A GPU is required, and training on all listings takes many hours.**
Install the extra packages with `pip install -r ../requirements-extra.txt`.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from datasets import Dataset
from pyvi import ViTokenizer
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error,
                             r2_score, root_mean_squared_error)
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          EarlyStoppingCallback, Trainer, TrainingArguments)

SEED = 42
MODEL_NAME = "vinai/phobert-base-v2"
MAX_LENGTH = 256   # PhoBERT cannot read more than 256 tokens

torch.cuda.is_available()

## Data

PhoBERT does not get the structured columns, so it reads the text **with its numbers** (area, bedrooms, ...).
The text comes from notebook 01; only the clauses that mention a price are removed.
The split is rebuilt exactly as in notebook 03 (sorted by date, 70/15/15), so the rows and indexes match the other notebooks.

In [ ]:
PRICE_CLAUSE = r"(?:^|(?<=[.,;!?\n]))[^.,;!?\n]*\b(?:giá|tỷ|tỉ|triệu|tr|vnd|vnđ|usd)\b[^.,;!?\n]*"

df = pd.read_parquet("../data/processed/priced_listings.parquet",
                     columns=["name", "description", "price", "published_at"])

# same split as notebook 03
df = df.sort_values("published_at").reset_index(drop=True)
n = len(df)
n_train, n_val = int(0.70 * n), int(0.15 * n)
df["split"] = np.repeat(["train", "val", "test"], [n_train, n_val, n - n_train - n_val])

text = (df["name"].fillna("") + ". " + df["description"].fillna("")).str.lower()
df = df.drop(columns=["name", "description"])
text = text.str.replace("[phone_number]", " ", regex=False)
text = text.str.replace(PRICE_CLAUSE, " ", regex=True)   # clauses that mention a price
df["text"] = text.str.replace(r"\s+", " ", regex=True).str.strip()
del text

train = df[df["split"] == "train"]
val = df[df["split"] == "val"]
test = df[df["split"] == "test"]

train["text"].head(3).tolist()

## Target

The model learns the **standardized** log price (mean 0, standard deviation 1, using training statistics).
A new regression head starts with outputs near 0, so this puts the target where the model already is.

In [ ]:
y_mean = np.log1p(train["price"]).mean()
y_std = np.log1p(train["price"]).std()

## Tokenization

1. **Word segmentation.** A Vietnamese word can have several syllables ("bất động sản"). PhoBERT was pretrained on text where those syllables are joined with `_` ("bất_động_sản"), so we prepare our text the same way with `pyvi`.
2. **Tokenizer.** The PhoBERT tokenizer turns the text into token ids and cuts it at 256 tokens.

Padding is done later, per batch, by `DataCollatorWithPadding`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def to_dataset(part):
    dataset = Dataset.from_dict({
        "text": [ViTokenizer.tokenize(text) for text in part["text"]],
        "label": ((np.log1p(part["price"]) - y_mean) / y_std).tolist(),
    })
    return dataset.map(lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH),
                       batched=True, remove_columns=["text"])


train_ds = to_dataset(train)
val_ds = to_dataset(val)
test_ds = to_dataset(test)

print(tokenizer.convert_ids_to_tokens(train_ds[0]["input_ids"][:30]))

## Model

`num_labels=1` adds a regression head with one output on top of PhoBERT and trains it with MSE loss.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)

## Training

| Setting | Why |
|---|---|
| small learning rate (`3e-5`) | large updates would destroy what PhoBERT already knows |
| warm-up (`warmup_ratio`) | the learning rate rises from 0 during the first 6% of steps, then decays linearly |
| `fp16` | half-precision on the GPU: about twice as fast, half the memory |
| evaluate and save every epoch | `load_best_model_at_end` restores the epoch with the lowest validation loss |
| `EarlyStoppingCallback` | stops when the validation loss gets worse |

If the GPU runs out of memory, lower `per_device_train_batch_size` to 16.

In [ ]:
args = TrainingArguments(
    output_dir="../models/phobert",
    num_train_epochs=3,
    learning_rate=3e-5,
    warmup_ratio=0.06,
    weight_decay=0.01,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=128,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=200,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)
trainer.train()

In [ ]:
log = pd.DataFrame(trainer.state.log_history)

ax = log.dropna(subset=["loss"]).plot(x="step", y="loss", label="train loss", figsize=(7, 3.5))
log.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", label="validation loss", marker="o", ax=ax)
ax.set_ylabel("MSE (standardized log price)")
plt.show()

## Results

Predictions are converted back: undo the standardization, then `expm1`.
The table also shows the results of notebook 06 on the same test rows
(note: the notebook 06 models also use the structured columns, PhoBERT uses the text only).

In [ ]:
def evaluate(y_true, y_pred):
    return {
        "MAE (bn VND)": mean_absolute_error(y_true, y_pred) / 1e9,
        "RMSE (bn VND)": root_mean_squared_error(y_true, y_pred) / 1e9,
        "R2": r2_score(y_true, y_pred),
        "MAPE (%)": 100 * mean_absolute_percentage_error(y_true, y_pred),
    }


# log1p(price) predicted by PhoBERT
phobert_val = trainer.predict(val_ds).predictions.squeeze(-1) * y_std + y_mean
phobert_test = trainer.predict(test_ds).predictions.squeeze(-1) * y_std + y_mean

table = pd.DataFrame({"PhoBERT (text only)": evaluate(test["price"], np.expm1(phobert_test))}).T

os.makedirs("../reports/metrics", exist_ok=True)
table.to_csv("../reports/metrics/08_phobert.csv")

notebook_06 = pd.read_csv("../reports/metrics/06_text_tfidf.csv", index_col=0)
pd.concat([notebook_06, table]).style.format("{:,.3f}")

## Save predictions

Notebook 10 combines the models, so it needs each model's prediction of `log1p(price)` for every validation and test row.
The index is the row number in `model_input.parquet`, which lets notebook 10 line up the predictions of different notebooks.

In [ ]:
predictions = pd.concat([
    pd.DataFrame({"split": "val", "PhoBERT": phobert_val}, index=val.index),
    pd.DataFrame({"split": "test", "PhoBERT": phobert_test}, index=test.index),
])

os.makedirs("../reports/predictions", exist_ok=True)
predictions.to_parquet("../reports/predictions/08_phobert.parquet")
predictions.head()